# 02 - IEEE 13-node unbalanced feeder

## Objective

Preserve single- and two-phase feeder topology and compare phase-specific voltage magnitudes from direct OpenDSS with the CEPT public CLI.

## Source, assumptions, and units

The source is the IEEE 13-node OpenDSS feeder bundled in the installed CEPT wheel. The comparison point is bus `671`; phase 1, 2, and 3 correspond to A, B, and C. The feeder's declared data are used unchanged. Voltage magnitude is line-to-neutral per unit (`pu`). This is a public demonstrator/reference feeder, not a user's physical project.

## Prediction

The three phase voltages at bus 671 will not be represented safely by one balanced number. The direct OpenDSS and CEPT values should agree within the teaching tolerance when both use the bundled source.

## Action

Solve the bundled source directly, then run `cept study demo unbalanced-load-flow` in a separate exact run directory. The CEPT command streams its output into the originating cell.

## Verification

Read the CEPT solver table from `results.json`, run `cept study verify` on that exact directory, and compare phase identities before comparing values.

## Interpretation

Phase-specific spread is an observation from this solver run. Agreement between two routes through the same public source is a workflow regression check, not independent validation.

## Exercise

Change `BUS_TO_INSPECT` to another named bus present in the direct feeder, inspect its three phase values, and state whether the phase spread increased or decreased. Rerun from a restarted kernel.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
# @title Setup — run once, then read the results below
import urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
exec(compile(_blob, "lesson helper", "exec"))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
lesson helpers ready: cli/read/table/cards + WORKSPACE.


### Direct reference - solve without CEPT

The bundled IEEE13 feeder is loaded straight into OpenDSS. `Solve` must report converged first.


In [2]:
MASTER_DSS = ieee13_master()
BUS_TO_INSPECT = '671'
import opendssdirect as dss

dss.Basic.ClearAll()
dss.Basic.DataPath(str(MASTER_DSS.parent))
dss.Text.Command(f'Redirect \"{MASTER_DSS}\"')
dss.Text.Command('CalcVoltageBases')
dss.Text.Command('Solve')
assert dss.Solution.Converged()


### Read bus 671 back

Three-phase voltage magnitude, solver-returned. Unbalanced laterals are the point of this feeder.


In [3]:
dss.Circuit.SetActiveBus(BUS_TO_INSPECT)
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}
table(['source', 'bus', 'phase', 'voltage magnitude', 'unit'], [('direct OpenDSS', BUS_TO_INSPECT, phase, value, 'pu') for phase, value in direct_by_phase.items()])
assert all(value > 0 for value in direct_by_phase.values())


| source | bus | phase | voltage magnitude | unit |
| --- | --- | --- | --- | --- |
| direct OpenDSS | 671 | 1 | 0.9827953877537872 | pu |
| direct OpenDSS | 671 | 2 | 1.040273943965177 | pu |
| direct OpenDSS | 671 | 3 | 0.9648999396286697 | pu |


### Run the same feeder through CEPT

`study run` persists solver artifacts under `runs/`; `study verify` checks the receipt. Both must pass before comparing.


In [4]:
RUN_DIR = WORKSPACE / 'runs' / '02-ieee13-unbalanced'
run_summary = cli('study', 'demo', 'unbalanced-load-flow', '--network', 'ieee13', '--out', RUN_DIR, '--force')
verify_summary = cli('study', 'verify', RUN_DIR)


$ cept study demo unbalanced-load-flow --network ieee13 --out '<notebook-workspace>\runs\02-ieee13-unbalanced' --force


→ exit 0


$ cept study verify '<notebook-workspace>\runs\02-ieee13-unbalanced'


→ exit 0


### Compare per phase

`results.json` carries the same solver quantities for bus 671. The difference column must sit near zero; the cards repeat the verdict.


In [5]:
results = read(RUN_DIR / 'results.json')
cept_rows = [row for row in results['load_flow']['bus_voltages'] if row['bus'].lower() == BUS_TO_INSPECT.lower()]
cept_by_phase = {row['phase']: row['v_pu'] for row in cept_rows}
table(['source', 'bus', 'phase', 'voltage magnitude', 'unit'], [('CEPT results.json', BUS_TO_INSPECT, row['phase'], row['v_pu'], 'pu') for row in cept_rows])
table(['phase', 'direct OpenDSS pu', 'CEPT pu', 'absolute difference pu'], [(phase, direct_by_phase[phase], cept_by_phase[phase], abs(direct_by_phase[phase] - cept_by_phase[phase])) for phase in (1, 2, 3)])

max_abs_diff_pu = max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3))
cards([
    ('CEPT run', run_summary['status'], 'solver-backed run artifacts'),
    ('Verify', str(verify_summary['passed']), 'receipt integrity and convergence'),
    ('Max |direct − CEPT|', f'{max_abs_diff_pu:.2e} pu', 'phase-voltage agreement'),
], title='2 · Unbalanced-feeder agreement')
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert set(cept_by_phase) == {1, 2, 3}
assert max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3)) < 1e-4


| source | bus | phase | voltage magnitude | unit |
| --- | --- | --- | --- | --- |
| CEPT results.json | 671 | 1 | 0.982797 | pu |
| CEPT results.json | 671 | 2 | 1.040275 | pu |
| CEPT results.json | 671 | 3 | 0.964889 | pu |
| phase | direct OpenDSS pu | CEPT pu | absolute difference pu |
| --- | --- | --- | --- |
| 1 | 0.9827953877537872 | 0.982797 | 1.6122462128675963e-06 |
| 2 | 1.040273943965177 | 1.040275 | 1.0560348231436478e-06 |
| 3 | 0.9648999396286697 | 0.964889 | 1.0939628669714985e-05 |


The table is built from the direct solver readback and the CEPT run's persisted `results.json`. Do not average the phases to hide imbalance. The verified public workflow remains a bounded `WORKFLOW_VALIDATED` result and does not establish project or field validation.